# Causal Robustness Analysis

Addresses two look-ahead issues from the audit:
1. **HMM decoding** — compare Viterbi (global) vs forward-filtered MAP (causal)
2. **Standardization** — compare full-sample z-score vs expanding-window z-score

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

sys.path.insert(0, str(Path.cwd().parent.parent))
from regime_utils import (
    RANDOM_STATE,
    expanding_zscore,
    forward_filter_labels,
    mean_regime_duration,
    read_csv,
)

In [ ]:
# Numerical helpers imported from regime_utils

In [ ]:
raw = read_csv("market_features_weekly.csv", parse_dates=["Date"], index_col="Date")
full_std = read_csv("market_features_weekly_std.csv", parse_dates=["Date"], index_col="Date")

model_features = raw[["SP500_Return", "VIX", "Yield_Spread"]]
X_full = full_std.values

exp_std = expanding_zscore(model_features)
X_exp = exp_std.values

print(f"Full-sample rows: {len(X_full)}")
print(f"Expanding-standardized rows (after {52}-week burn-in): {len(X_exp)}")

## 1. Decoding comparison (full-sample standardized features)

In [ ]:
hmm = GaussianHMM(
    n_components=3, covariance_type="full", n_iter=1000, random_state=RANDOM_STATE
)
hmm.fit(X_full)

gmm = GaussianMixture(
    n_components=3, covariance_type="full", n_init=10, random_state=RANDOM_STATE
)
gmm.fit(X_full)

viterbi = hmm.predict(X_full)
filtered = forward_filter_labels(hmm, X_full)
_, posteriors = hmm.score_samples(X_full)
smoothed_map = posteriors.argmax(axis=1)
gmm_labels = gmm.predict(X_full)

decode_results = pd.DataFrame({
    "Method": ["GMM (pointwise)", "HMM Viterbi (global)", "HMM smoothed MAP", "HMM forward-filtered (causal)"],
    "Mean duration (weeks)": [
        mean_regime_duration(gmm_labels),
        mean_regime_duration(viterbi),
        mean_regime_duration(smoothed_map),
        mean_regime_duration(filtered),
    ],
})
decode_results["Mean duration (weeks)"] = decode_results["Mean duration (weeks)"].round(1)
print(decode_results.to_string(index=False))
print(f"\nViterbi vs filtered label agreement: {(viterbi == filtered).mean():.1%}")

## 2. Expanding standardization + refit

In [ ]:
hmm_exp = GaussianHMM(
    n_components=3, covariance_type="full", n_iter=1000, random_state=RANDOM_STATE
)
hmm_exp.fit(X_exp)

gmm_exp = GaussianMixture(
    n_components=3, covariance_type="full", n_init=10, random_state=RANDOM_STATE
)
gmm_exp.fit(X_exp)

viterbi_exp = hmm_exp.predict(X_exp)
filtered_exp = forward_filter_labels(hmm_exp, X_exp)
gmm_exp_labels = gmm_exp.predict(X_exp)

exp_results = pd.DataFrame({
    "Method": ["GMM (pointwise)", "HMM Viterbi", "HMM forward-filtered"],
    "Full-sample std duration": [
        mean_regime_duration(gmm_labels),
        mean_regime_duration(viterbi),
        mean_regime_duration(filtered),
    ],
    "Expanding std duration": [
        mean_regime_duration(gmm_exp_labels),
        mean_regime_duration(viterbi_exp),
        mean_regime_duration(filtered_exp),
    ],
})
exp_results = exp_results.round(1)
print(exp_results.to_string(index=False))

## 3. Summary — headline finding

The duration gap narrows substantially when HMM uses causal (forward-filtered) decoding instead of Viterbi. The remaining gap vs GMM reflects HMM transition structure, not smoothing look-ahead alone.

In [ ]:
viterbi_dur = mean_regime_duration(viterbi)
filtered_dur = mean_regime_duration(filtered)
gmm_dur = mean_regime_duration(gmm_labels)

print(f"Original headline:  HMM Viterbi {viterbi_dur:.1f} wks vs GMM {gmm_dur:.1f} wks")
print(f"Causal comparison:  HMM filtered {filtered_dur:.1f} wks vs GMM {gmm_dur:.1f} wks")
print(f"Smoothing share:      ~{(1 - filtered_dur / viterbi_dur):.0%} of Viterbi persistence was decoding look-ahead")
print(f"\nCross-model ARI (full-sample std, seed {RANDOM_STATE}): {adjusted_rand_score(gmm_labels, viterbi):.2f}")
print(f"Cross-model ARI (filtered HMM vs GMM):                  {adjusted_rand_score(gmm_labels, filtered):.2f}")